In [1]:
# ============================================================
# STEP 1: DATASET PROFILING
# Starbucks vs McDonald's Nutrition Project
# ============================================================

# ------------------------------------------------------------
# 1. Spark Setup for Windows/Jupyter
# ------------------------------------------------------------

import os

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, when

spark = SparkSession.builder \
    .appName("Starbucks_McDonalds_Dataset_Profiling") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print("Spark Started Successfully!")

# ------------------------------------------------------------
# 2. CSV File Paths
# ------------------------------------------------------------

mcd_path = r"C:\Users\visva\Desktop\Datamites\Datamites capstone projects\Prce_002_Snack Analysis\csv\mcd.csv"

sb_expanded_path = r"C:\Users\visva\Desktop\Datamites\Datamites capstone projects\Prce_002_Snack Analysis\csv\sb_drink_expanded.csv"

sb_drinks_path = r"C:\Users\visva\Desktop\Datamites\Datamites capstone projects\Prce_002_Snack Analysis\csv\sb_drinks.csv"

sb_food_path = r"C:\Users\visva\Desktop\Datamites\Datamites capstone projects\Prce_002_Snack Analysis\csv\sb_food.csv"

# ------------------------------------------------------------
# 3. Read CSV Files
# ------------------------------------------------------------

mcd_df = spark.read.csv(mcd_path, header=True, inferSchema=True)
sb_expanded_df = spark.read.csv(sb_expanded_path, header=True, inferSchema=True)
sb_drinks_df = spark.read.csv(sb_drinks_path, header=True, inferSchema=True)
sb_food_df = spark.read.csv(sb_food_path, header=True, inferSchema=True)

# ------------------------------------------------------------
# 4. Store All Datasets Together
# ------------------------------------------------------------

datasets = {
    "McDonalds Menu": mcd_df,
    "Starbucks Expanded Drinks": sb_expanded_df,
    "Starbucks Nutrition Drinks": sb_drinks_df,
    "Starbucks Food": sb_food_df
}

# ------------------------------------------------------------
# 5. Basic Dataset Information
# ------------------------------------------------------------

print("\n================ BASIC DATASET INFORMATION ================\n")

for name, df in datasets.items():
    print("\n------------------------------------------------------------")
    print(name)
    print("------------------------------------------------------------")
    print("Rows:", df.count())
    print("Columns:", len(df.columns))
    print("Column Names:")
    print(df.columns)

# ------------------------------------------------------------
# 6. Schema Check
# ------------------------------------------------------------

print("\n================ SCHEMA CHECK ================\n")

for name, df in datasets.items():
    print("\n------------------------------------------------------------")
    print(f"Schema: {name}")
    print("------------------------------------------------------------")
    df.printSchema()

# ------------------------------------------------------------
# 7. Sample Records
# ------------------------------------------------------------

print("\n================ SAMPLE RECORDS ================\n")

for name, df in datasets.items():
    print("\n------------------------------------------------------------")
    print(f"Sample Records: {name}")
    print("------------------------------------------------------------")
    df.show(5, truncate=False)

# ------------------------------------------------------------
# 8. Full Duplicate Row Check
# ------------------------------------------------------------

print("\n================ FULL DUPLICATE ROW CHECK ================\n")

for name, df in datasets.items():
    total_rows = df.count()
    distinct_rows = df.dropDuplicates().count()
    duplicate_rows = total_rows - distinct_rows

    print("\n------------------------------------------------------------")
    print(name)
    print("------------------------------------------------------------")
    print("Total Rows:", total_rows)
    print("Distinct Rows:", distinct_rows)
    print("Duplicate Rows:", duplicate_rows)

# ------------------------------------------------------------
# 9. Show Duplicate Rows in Starbucks Nutrition Drinks
# Backticks are used because column names contain dots/spaces/brackets
# ------------------------------------------------------------

print("\n================ DUPLICATE ROWS: STARBUCKS NUTRITION DRINKS ================\n")

duplicate_sb_drinks = sb_drinks_df.groupBy(
    *[col(f"`{c}`") for c in sb_drinks_df.columns]
).count().filter(col("count") > 1)

duplicate_sb_drinks.show(truncate=False)

# ------------------------------------------------------------
# 10. Null and '-' Value Check
# ------------------------------------------------------------

print("\n================ NULL AND '-' VALUE CHECK ================\n")

for name, df in datasets.items():
    print("\n------------------------------------------------------------")
    print(f"Missing / '-' Value Check: {name}")
    print("------------------------------------------------------------")

    missing_exprs = [
        spark_sum(
            when(
                col(f"`{c}`").isNull() | (col(f"`{c}`").cast("string") == "-"),
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]

    df.select(missing_exprs).show(truncate=False)

# ------------------------------------------------------------
# 11. End of Step 1
# ------------------------------------------------------------

print("\n============================================================")
print("STEP 1 DATASET PROFILING COMPLETED SUCCESSFULLY")
print("============================================================")

## ============================================================
# STEP 2: DATA CLEANING & STANDARDIZATION - FINAL CORRECTED
# ============================================================

from pyspark.sql.functions import col, trim, when, sum as spark_sum
from pyspark.sql.types import StringType

# ------------------------------------------------------------
# 1. Helper Function: Trim Text Columns
# ------------------------------------------------------------

def trim_string_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, trim(col(f"`{field.name}`")))
    return df

# ------------------------------------------------------------
# 2. Helper Function: Replace '-' and blank strings with NULL
# ------------------------------------------------------------

def replace_missing_placeholders(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(
                field.name,
                when(
                    (trim(col(f"`{field.name}`")) == "-") |
                    (trim(col(f"`{field.name}`")) == ""),
                    None
                ).otherwise(trim(col(f"`{field.name}`")))
            )
    return df

# ------------------------------------------------------------
# 3. Helper Function: Convert Selected Columns to Double
# ------------------------------------------------------------

def convert_columns_to_double(df, column_list):
    for c in column_list:
        if c in df.columns:
            df = df.withColumn(c, col(f"`{c}`").cast("double"))
    return df

# ------------------------------------------------------------
# 4. Clean McDonald's Dataset
# ------------------------------------------------------------

mcd_clean_df = trim_string_columns(mcd_df)
mcd_clean_df = replace_missing_placeholders(mcd_clean_df)

mcd_numeric_columns = [
    "Calories", "Calories from Fat", "Total Fat",
    "Total Fat (% Daily Value)", "Saturated Fat",
    "Saturated Fat (% Daily Value)", "Trans Fat",
    "Cholesterol", "Cholesterol (% Daily Value)",
    "Sodium", "Sodium (% Daily Value)",
    "Carbohydrates", "Carbohydrates (% Daily Value)",
    "Dietary Fiber", "Dietary Fiber (% Daily Value)",
    "Sugars", "Protein",
    "Vitamin A (% Daily Value)", "Vitamin C (% Daily Value)",
    "Calcium (% Daily Value)", "Iron (% Daily Value)"
]

mcd_clean_df = convert_columns_to_double(mcd_clean_df, mcd_numeric_columns)

# ------------------------------------------------------------
# 5. Clean Starbucks Expanded Drinks Dataset
# Keep Caffeine (mg) as text because it has numbers + Varies
# ------------------------------------------------------------

sb_expanded_clean_df = trim_string_columns(sb_expanded_df)
sb_expanded_clean_df = replace_missing_placeholders(sb_expanded_clean_df)

# Standardize varies/Varies
sb_expanded_clean_df = sb_expanded_clean_df.withColumn(
    "Caffeine (mg)",
    when(col("`Caffeine (mg)`").isin("Varies", "varies"), "Varies")
    .otherwise(col("`Caffeine (mg)`"))
)

# Fix corrupted Total Fat value
sb_expanded_clean_df = sb_expanded_clean_df.withColumn(
    "Total Fat (g)",
    when(
        col("`Total Fat (g)`").cast("string") == "2024-03-02 00:00:00",
        "3.2"
    ).otherwise(col("`Total Fat (g)`").cast("string"))
)

sb_expanded_numeric_columns = [
    "Calories", "Total Fat (g)", "Trans Fat (g)",
    "Saturated Fat (g)", "Sodium (mg)",
    "Total Carbohydrates (g)", "Cholesterol (mg)",
    "Dietary Fibre (g)", "Sugars (g)", "Protein (g)",
    "Vitamin A (% DV)", "Vitamin C (% DV)",
    "Calcium (% DV)", "Iron (% DV)"
]

sb_expanded_clean_df = convert_columns_to_double(
    sb_expanded_clean_df,
    sb_expanded_numeric_columns
)

# ------------------------------------------------------------
# 6. Clean Starbucks Nutrition Drinks Dataset
# ------------------------------------------------------------

sb_drinks_clean_df = sb_drinks_df.withColumnRenamed("Unnamed: 0", "Item_Name")
sb_drinks_clean_df = trim_string_columns(sb_drinks_clean_df)
sb_drinks_clean_df = replace_missing_placeholders(sb_drinks_clean_df)

sb_drinks_numeric_columns = [
    "Calories", "Fat (g)", "Carb. (g)",
    "Fiber (g)", "Protein", "Sodium"
]

sb_drinks_clean_df = convert_columns_to_double(
    sb_drinks_clean_df,
    sb_drinks_numeric_columns
)

sb_drinks_before_dedup = sb_drinks_clean_df.count()
sb_drinks_clean_df = sb_drinks_clean_df.dropDuplicates()
sb_drinks_after_dedup = sb_drinks_clean_df.count()

print("Starbucks Nutrition Drinks rows before duplicate removal:", sb_drinks_before_dedup)
print("Starbucks Nutrition Drinks rows after duplicate removal:", sb_drinks_after_dedup)
print("Duplicate rows removed:", sb_drinks_before_dedup - sb_drinks_after_dedup)

# ------------------------------------------------------------
# 7. Clean Starbucks Food Dataset
# ------------------------------------------------------------

sb_food_clean_df = sb_food_df.withColumnRenamed("Unnamed: 0", "Item_Name")
sb_food_clean_df = trim_string_columns(sb_food_clean_df)
sb_food_clean_df = replace_missing_placeholders(sb_food_clean_df)

sb_food_numeric_columns = [
    "Calories", "Fat (g)", "Carb. (g)",
    "Fiber (g)", "Protein (g)"
]

sb_food_clean_df = convert_columns_to_double(
    sb_food_clean_df,
    sb_food_numeric_columns
)

# ------------------------------------------------------------
# 8. Verify Cleaned Datasets
# ------------------------------------------------------------

cleaned_datasets = {
    "McDonalds Clean": mcd_clean_df,
    "Starbucks Expanded Drinks Clean": sb_expanded_clean_df,
    "Starbucks Nutrition Drinks Clean": sb_drinks_clean_df,
    "Starbucks Food Clean": sb_food_clean_df
}

print("\n================ CLEANED DATASET SCHEMAS ================\n")

for name, df in cleaned_datasets.items():
    print("\n------------------------------------------------------------")
    print(name)
    print("------------------------------------------------------------")
    df.printSchema()

# ------------------------------------------------------------
# 9. Null Value Check After Cleaning
# ------------------------------------------------------------

print("\n================ NULL VALUE CHECK AFTER CLEANING ================\n")

for name, df in cleaned_datasets.items():
    print("\n------------------------------------------------------------")
    print(f"Null Check: {name}")
    print("------------------------------------------------------------")

    null_exprs = [
        spark_sum(when(col(f"`{c}`").isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]

    df.select(null_exprs).show(truncate=False)

# ------------------------------------------------------------
# 10. Special Value Checks
# ------------------------------------------------------------

print("\n================ STARBUCKS EXPANDED CHECKS ================\n")

print("Caffeine values after cleaning:")
sb_expanded_clean_df.groupBy("Caffeine (mg)").count().orderBy("Caffeine (mg)").show(100, truncate=False)

print("Total Fat values after cleaning:")
sb_expanded_clean_df.select("Total Fat (g)").distinct().orderBy("Total Fat (g)").show(100, truncate=False)

print("Corrected Total Fat row:")
sb_expanded_clean_df.filter(
    (col("`Beverage_category`") == "Frappuccino® Blended Crème") &
    (col("`Beverage`") == "Strawberries & Crème (Without Whipped Cream)") &
    (col("`Beverage_prep`") == "Soymilk") &
    (col("`Calories`") == 320.0)
).show(truncate=False)

# ------------------------------------------------------------
# 11. End of Step 2
# ------------------------------------------------------------

print("\n============================================================")
print("STEP 2 DATA CLEANING & STANDARDIZATION COMPLETED SUCCESSFULLY")
print("============================================================")


# =======================================================================================
# STEP 3: COMMON SCHEMA CREATION
# Starbucks vs McDonald's Nutrition Project
# ============================================================

from pyspark.sql.functions import col, lit, concat_ws, when, sum as spark_sum

# ------------------------------------------------------------
# 1. Create McDonald's Standard DataFrame
# ------------------------------------------------------------

mcd_standard_df = mcd_clean_df.select(
    lit("McDonalds").alias("Brand"),
    lit("mcd.csv").alias("Source_File"),
    col("`Category`").alias("Category"),
    col("`Item`").alias("Item_Name"),
    col("`Calories`").alias("Calories"),
    col("`Total Fat`").alias("Fat"),
    col("`Carbohydrates`").alias("Carbohydrates"),
    col("`Dietary Fiber`").alias("Fiber"),
    col("`Sugars`").alias("Sugars"),
    col("`Protein`").alias("Protein"),
    col("`Sodium`").alias("Sodium"),
    lit(None).cast("string").alias("Caffeine")
)

# ------------------------------------------------------------
# 2. Create Starbucks Expanded Drinks Standard DataFrame
# ------------------------------------------------------------

sb_expanded_standard_df = sb_expanded_clean_df.select(
    lit("Starbucks").alias("Brand"),
    lit("sb_drink_expanded.csv").alias("Source_File"),
    col("`Beverage_category`").alias("Category"),
    concat_ws(" - ", col("`Beverage`"), col("`Beverage_prep`")).alias("Item_Name"),
    col("`Calories`").alias("Calories"),
    col("`Total Fat (g)`").alias("Fat"),
    col("`Total Carbohydrates (g)`").alias("Carbohydrates"),
    col("`Dietary Fibre (g)`").alias("Fiber"),
    col("`Sugars (g)`").alias("Sugars"),
    col("`Protein (g)`").alias("Protein"),
    col("`Sodium (mg)`").alias("Sodium"),
    col("`Caffeine (mg)`").cast("string").alias("Caffeine")
)

# ------------------------------------------------------------
# 3. Create Starbucks Nutrition Drinks Standard DataFrame
# Missing Sugars and Caffeine are kept as NULL
# ------------------------------------------------------------

sb_drinks_standard_df = sb_drinks_clean_df.select(
    lit("Starbucks").alias("Brand"),
    lit("sb_drinks.csv").alias("Source_File"),
    lit("Drink").alias("Category"),
    col("`Item_Name`").alias("Item_Name"),
    col("`Calories`").alias("Calories"),
    col("`Fat (g)`").alias("Fat"),
    col("`Carb. (g)`").alias("Carbohydrates"),
    col("`Fiber (g)`").alias("Fiber"),
    lit(None).cast("double").alias("Sugars"),
    col("`Protein`").alias("Protein"),
    col("`Sodium`").alias("Sodium"),
    lit(None).cast("string").alias("Caffeine")
)

# ------------------------------------------------------------
# 4. Create Starbucks Food Standard DataFrame
# Missing Sugars, Sodium, and Caffeine are kept as NULL
# ------------------------------------------------------------

sb_food_standard_df = sb_food_clean_df.select(
    lit("Starbucks").alias("Brand"),
    lit("sb_food.csv").alias("Source_File"),
    lit("Food").alias("Category"),
    col("`Item_Name`").alias("Item_Name"),
    col("`Calories`").alias("Calories"),
    col("`Fat (g)`").alias("Fat"),
    col("`Carb. (g)`").alias("Carbohydrates"),
    col("`Fiber (g)`").alias("Fiber"),
    lit(None).cast("double").alias("Sugars"),
    col("`Protein (g)`").alias("Protein"),
    lit(None).cast("double").alias("Sodium"),
    lit(None).cast("string").alias("Caffeine")
)

# ------------------------------------------------------------
# 5. Union All Standardized DataFrames
# ------------------------------------------------------------

master_nutrition_df = mcd_standard_df \
    .unionByName(sb_expanded_standard_df) \
    .unionByName(sb_drinks_standard_df) \
    .unionByName(sb_food_standard_df)

# ------------------------------------------------------------
# 6. Verify Master Dataset
# ------------------------------------------------------------

print("\n================ MASTER DATASET SCHEMA ================\n")
master_nutrition_df.printSchema()

print("\n================ MASTER DATASET ROW COUNT ================\n")
print("Total rows:", master_nutrition_df.count())

print("\n================ MASTER DATASET BRAND COUNTS ================\n")
master_nutrition_df.groupBy("Brand").count().show()

print("\n================ MASTER DATASET SOURCE FILE COUNTS ================\n")
master_nutrition_df.groupBy("Source_File").count().show(truncate=False)

print("\n================ MASTER DATASET SAMPLE RECORDS ================\n")
master_nutrition_df.show(20, truncate=False)

# ------------------------------------------------------------
# 7. Null Check on Master Dataset
# ------------------------------------------------------------

print("\n================ MASTER DATASET NULL CHECK ================\n")

null_exprs = [
    spark_sum(
        when(col(f"`{c}`").isNull(), 1).otherwise(0)
    ).alias(c)
    for c in master_nutrition_df.columns
]

master_nutrition_df.select(null_exprs).show(truncate=False)

# ------------------------------------------------------------
# 8. End of Step 3
# ------------------------------------------------------------

print("\n============================================================")
print("STEP 3 COMMON SCHEMA CREATION COMPLETED SUCCESSFULLY")
print("============================================================")


# ============================================================
# STEP 4: EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

from pyspark.sql.functions import avg, round, count, when, sum as spark_sum

# ------------------------------------------------------------
# 1. Total Records
# ------------------------------------------------------------

print("\n================ TOTAL RECORDS ================\n")

print("Total Records:", master_nutrition_df.count())

# ------------------------------------------------------------
# 2. Records by Brand
# ------------------------------------------------------------

print("\n================ RECORDS BY BRAND ================\n")

master_nutrition_df.groupBy("Brand") \
    .count() \
    .show(truncate=False)

# ------------------------------------------------------------
# 3. Records by Source File
# ------------------------------------------------------------

print("\n================ RECORDS BY SOURCE FILE ================\n")

master_nutrition_df.groupBy("Source_File") \
    .count() \
    .show(truncate=False)

# ------------------------------------------------------------
# 4. Records by Category
# ------------------------------------------------------------

print("\n================ RECORDS BY CATEGORY ================\n")

master_nutrition_df.groupBy("Category") \
    .count() \
    .orderBy("Category") \
    .show(100, truncate=False)

# ------------------------------------------------------------
# 5. Average Nutrition by Brand
# ------------------------------------------------------------

print("\n================ AVERAGE NUTRITION BY BRAND ================\n")

master_nutrition_df.groupBy("Brand").agg(
    round(avg("Calories"),2).alias("Avg_Calories"),
    round(avg("Fat"),2).alias("Avg_Fat"),
    round(avg("Carbohydrates"),2).alias("Avg_Carbohydrates"),
    round(avg("Fiber"),2).alias("Avg_Fiber"),
    round(avg("Sugars"),2).alias("Avg_Sugars"),
    round(avg("Protein"),2).alias("Avg_Protein"),
    round(avg("Sodium"),2).alias("Avg_Sodium")
).show(truncate=False)

# ------------------------------------------------------------
# 6. Average Nutrition by Source File
# ------------------------------------------------------------

print("\n================ AVERAGE NUTRITION BY SOURCE FILE ================\n")

master_nutrition_df.groupBy("Source_File").agg(
    round(avg("Calories"),2).alias("Avg_Calories"),
    round(avg("Fat"),2).alias("Avg_Fat"),
    round(avg("Carbohydrates"),2).alias("Avg_Carbohydrates"),
    round(avg("Fiber"),2).alias("Avg_Fiber"),
    round(avg("Sugars"),2).alias("Avg_Sugars"),
    round(avg("Protein"),2).alias("Avg_Protein"),
    round(avg("Sodium"),2).alias("Avg_Sodium")
).show(truncate=False)

# ------------------------------------------------------------
# 7. Non-Null Value Counts by Brand
# ------------------------------------------------------------

print("\n================ NON-NULL VALUE COUNTS BY BRAND ================\n")

master_nutrition_df.groupBy("Brand").agg(
    count("Calories").alias("Calories_Count"),
    count("Fat").alias("Fat_Count"),
    count("Carbohydrates").alias("Carbohydrates_Count"),
    count("Fiber").alias("Fiber_Count"),
    count("Sugars").alias("Sugars_Count"),
    count("Protein").alias("Protein_Count"),
    count("Sodium").alias("Sodium_Count")
).show(truncate=False)

# ------------------------------------------------------------
# 8. Missing Value Counts
# ------------------------------------------------------------

print("\n================ MISSING VALUES BY SOURCE FILE ================\n")

master_nutrition_df.groupBy("Source_File").agg(
    spark_sum(
        when(col("Calories").isNull(), 1).otherwise(0)
    ).alias("Missing_Calories"),

    spark_sum(
        when(col("Fat").isNull(), 1).otherwise(0)
    ).alias("Missing_Fat"),

    spark_sum(
        when(col("Carbohydrates").isNull(), 1).otherwise(0)
    ).alias("Missing_Carbohydrates"),

    spark_sum(
        when(col("Fiber").isNull(), 1).otherwise(0)
    ).alias("Missing_Fiber"),

    spark_sum(
        when(col("Sugars").isNull(), 1).otherwise(0)
    ).alias("Missing_Sugars"),

    spark_sum(
        when(col("Protein").isNull(), 1).otherwise(0)
    ).alias("Missing_Protein"),

    spark_sum(
        when(col("Sodium").isNull(), 1).otherwise(0)
    ).alias("Missing_Sodium"),

    spark_sum(
        when(col("Caffeine").isNull(), 1).otherwise(0)
    ).alias("Missing_Caffeine")

).show(truncate=False)
# ------------------------------------------------------------
# 9. End of Step 4
# ------------------------------------------------------------

print("\n============================================================")
print("STEP 4 EDA COMPLETED SUCCESSFULLY")
print("============================================================")



# ============================================================
# STEP 5: CATEGORY-BASED HEALTH COMPARISON
# McDonald's Drinks vs Starbucks Drinks
# McDonald's Food vs Starbucks Food
# ============================================================

from pyspark.sql.functions import col, when, avg, round, count

# ------------------------------------------------------------
# 1. Create Product_Type column
# ------------------------------------------------------------

comparison_df = master_nutrition_df.withColumn(
    "Product_Type",
    when(
        (col("Brand") == "McDonalds") &
        (col("Category").isin("Beverages", "Coffee & Tea", "Smoothies & Shakes")),
        "Drink"
    ).when(
        (col("Brand") == "McDonalds") &
        (~col("Category").isin("Beverages", "Coffee & Tea", "Smoothies & Shakes")),
        "Food"
    ).when(
        (col("Brand") == "Starbucks") &
        (col("Source_File").isin("sb_drink_expanded.csv", "sb_drinks.csv")),
        "Drink"
    ).when(
        (col("Brand") == "Starbucks") &
        (col("Source_File") == "sb_food.csv"),
        "Food"
    ).otherwise("Other")
)

# ------------------------------------------------------------
# 2. Check Product Type Counts
# ------------------------------------------------------------

print("\n================ PRODUCT TYPE COUNTS ================\n")

comparison_df.groupBy("Brand", "Product_Type") \
    .count() \
    .orderBy("Brand", "Product_Type") \
    .show(truncate=False)

# ------------------------------------------------------------
# 3. Drinks Comparison
# McDonald's Drinks vs Starbucks Drinks
# ------------------------------------------------------------

print("\n================ DRINKS: MCDONALD'S VS STARBUCKS ================\n")

comparison_df.filter(
    col("Product_Type") == "Drink"
).groupBy("Brand").agg(
    count("*").alias("Total_Items"),
    count("Calories").alias("Calories_Count"),
    round(avg("Calories"), 2).alias("Avg_Calories"),
    round(avg("Fat"), 2).alias("Avg_Fat"),
    round(avg("Carbohydrates"), 2).alias("Avg_Carbohydrates"),
    round(avg("Fiber"), 2).alias("Avg_Fiber"),
    count("Sugars").alias("Sugars_Count"),
    round(avg("Sugars"), 2).alias("Avg_Sugars"),
    round(avg("Protein"), 2).alias("Avg_Protein"),
    count("Sodium").alias("Sodium_Count"),
    round(avg("Sodium"), 2).alias("Avg_Sodium")
).show(truncate=False)

# ------------------------------------------------------------
# 4. Food Comparison
# McDonald's Food vs Starbucks Food
# ------------------------------------------------------------

print("\n================ FOOD: MCDONALD'S VS STARBUCKS ================\n")

comparison_df.filter(
    col("Product_Type") == "Food"
).groupBy("Brand").agg(
    count("*").alias("Total_Items"),
    count("Calories").alias("Calories_Count"),
    round(avg("Calories"), 2).alias("Avg_Calories"),
    round(avg("Fat"), 2).alias("Avg_Fat"),
    round(avg("Carbohydrates"), 2).alias("Avg_Carbohydrates"),
    round(avg("Fiber"), 2).alias("Avg_Fiber"),
    count("Sugars").alias("Sugars_Count"),
    round(avg("Sugars"), 2).alias("Avg_Sugars"),
    round(avg("Protein"), 2).alias("Avg_Protein"),
    count("Sodium").alias("Sodium_Count"),
    round(avg("Sodium"), 2).alias("Avg_Sodium")
).show(truncate=False)

# ------------------------------------------------------------
# 5. Overall Brand Comparison
# Supporting comparison only
# ------------------------------------------------------------

print("\n================ OVERALL BRAND COMPARISON ================\n")

comparison_df.groupBy("Brand").agg(
    count("*").alias("Total_Items"),
    round(avg("Calories"), 2).alias("Avg_Calories"),
    round(avg("Fat"), 2).alias("Avg_Fat"),
    round(avg("Carbohydrates"), 2).alias("Avg_Carbohydrates"),
    round(avg("Fiber"), 2).alias("Avg_Fiber"),
    count("Sugars").alias("Sugars_Count"),
    round(avg("Sugars"), 2).alias("Avg_Sugars"),
    round(avg("Protein"), 2).alias("Avg_Protein"),
    count("Sodium").alias("Sodium_Count"),
    round(avg("Sodium"), 2).alias("Avg_Sodium")
).show(truncate=False)

# ------------------------------------------------------------
# 6. Highest Calorie Items
# ------------------------------------------------------------

print("\n================ HIGHEST CALORIE ITEMS ================\n")

comparison_df.filter(
    col("Calories").isNotNull()
).select(
    "Brand", "Product_Type", "Category", "Item_Name", "Calories"
).orderBy(
    col("Calories").desc()
).show(20, truncate=False)

# ------------------------------------------------------------
# 7. Lowest Calorie Items
# ------------------------------------------------------------

print("\n================ LOWEST CALORIE ITEMS ================\n")

comparison_df.filter(
    col("Calories").isNotNull()
).select(
    "Brand", "Product_Type", "Category", "Item_Name", "Calories"
).orderBy(
    col("Calories").asc()
).show(20, truncate=False)

# ------------------------------------------------------------
# 8. Highest Sugar Items
# ------------------------------------------------------------

print("\n================ HIGHEST SUGAR ITEMS ================\n")

comparison_df.filter(
    col("Sugars").isNotNull()
).select(
    "Brand", "Product_Type", "Category", "Item_Name", "Sugars"
).orderBy(
    col("Sugars").desc()
).show(20, truncate=False)

# ------------------------------------------------------------
# 9. Lowest Sugar Items
# ------------------------------------------------------------

print("\n================ LOWEST SUGAR ITEMS ================\n")

comparison_df.filter(
    col("Sugars").isNotNull()
).select(
    "Brand", "Product_Type", "Category", "Item_Name", "Sugars"
).orderBy(
    col("Sugars").asc()
).show(20, truncate=False)

# ------------------------------------------------------------
# 10. Highest Sodium Items
# ------------------------------------------------------------

print("\n================ HIGHEST SODIUM ITEMS ================\n")

comparison_df.filter(
    col("Sodium").isNotNull()
).select(
    "Brand", "Product_Type", "Category", "Item_Name", "Sodium"
).orderBy(
    col("Sodium").desc()
).show(20, truncate=False)

# ------------------------------------------------------------
# 11. Lowest Sodium Items
# ------------------------------------------------------------

print("\n================ LOWEST SODIUM ITEMS ================\n")

comparison_df.filter(
    col("Sodium").isNotNull()
).select(
    "Brand", "Product_Type", "Category", "Item_Name", "Sodium"
).orderBy(
    col("Sodium").asc()
).show(20, truncate=False)

# ------------------------------------------------------------
# 12. Highest Protein Items
# Higher protein can be considered positive in nutrition comparison
# ------------------------------------------------------------

print("\n================ HIGHEST PROTEIN ITEMS ================\n")

comparison_df.filter(
    col("Protein").isNotNull()
).select(
    "Brand", "Product_Type", "Category", "Item_Name", "Protein"
).orderBy(
    col("Protein").desc()
).show(20, truncate=False)


# ------------------------------------------------------------
# 14. End of Step 5
# ------------------------------------------------------------

print("\n============================================================")
print("STEP 5 CATEGORY-BASED HEALTH COMPARISON COMPLETED SUCCESSFULLY")
print("============================================================")

Spark Started Successfully!

================ BASIC DATASET INFORMATION ================


------------------------------------------------------------
McDonalds Menu
------------------------------------------------------------
Rows: 260
Columns: 24
Column Names:
['Category', 'Item', 'Serving Size', 'Calories', 'Calories from Fat', 'Total Fat', 'Total Fat (% Daily Value)', 'Saturated Fat', 'Saturated Fat (% Daily Value)', 'Trans Fat', 'Cholesterol', 'Cholesterol (% Daily Value)', 'Sodium', 'Sodium (% Daily Value)', 'Carbohydrates', 'Carbohydrates (% Daily Value)', 'Dietary Fiber', 'Dietary Fiber (% Daily Value)', 'Sugars', 'Protein', 'Vitamin A (% Daily Value)', 'Vitamin C (% Daily Value)', 'Calcium (% Daily Value)', 'Iron (% Daily Value)']

------------------------------------------------------------
Starbucks Expanded Drinks
------------------------------------------------------------
Rows: 242
Columns: 18
Column Names:
['Beverage_category', 'Beverage', 'Beverage_prep', 'Calories', '